## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
#!CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 235.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 294.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 212.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 294.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
# For installing the libraries & downloading models from HF Hub
!pip install --upgrade pip setuptools wheel -q

# Install numpy separately first to prevent build issues, using a version compatible with tensorflow
!pip install numpy==1.26.4 -q

# Then install the rest of the libraries
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but 

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

# **To Do:**

* 1. Load the large language model from Hugging Face
* 2. Create a function to define the model parameters and generate a response
*3.  Apply the response generation function to get answers to the questions provided in the problem statement ( 5 questions)


In [2]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"


In [3]:
# Using hf_hub_download to download a model from the Hugging Face model hub
# The repo_id parameter specifies the model name or path in the Hugging Face repository
# The filename parameter specifies the name of the file to download
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

In [4]:
llm = Llama(
    model_path=model_path,
    n_ctx=2300,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [5]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [6]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
response(user_input)

'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. ABCs'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [7]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure that extends from the large intestine. The symptoms of appendicitis can vary from person to person, but some common signs include:\n\n1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be constant or come and go. It may start as a mild discomfort that worsens over time.\n2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and nausea'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [8]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications."

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [9]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nA person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:\n\n1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek emergency medical attention as soon as possible. The primary goal of emergency care is to prevent further damage to the brain, stabilize vital signs, and manage any life-threatening conditions.\n2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various conditions associated with a'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [10]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint, sl"

In [11]:
user_input = "Does iPhone usage cause sepsis?"
response(user_input)

Llama.generate: prefix-match hit


" No, there is no scientific evidence to support the claim that using an iPhone or any other smartphone can cause sepsis. Sepsis is a serious condition caused by the body's response to infection, and it requires medical attention if suspected. Using a smartphone does not increase the risk of developing sepsis."

Observation/Inference ⁉
1. From the above answers, it is evident that the LLM Model has knowledge  to answer the question, but never answered that "it ws not aware". It could be hallucinating in the response, it is to be verified further.
2. There is no details of the percentage of truthfulness from the model's response.

## Question Answering using LLM with Prompt Engineering

To Do:

1. Setting the system prompt here , to set the overall behavior , persona and constraints for the AI.

In [12]:

system_prompt = """Act as a highly knowledgeable and helpful AI medical assistant. Your primary goal is to provide accurate, comprehensive response relevant to the ask. If not sure, feel free to say 'I dont know'. Context: {context}

Question: {question}

Answer:"""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [13]:
user_input = system_prompt+"\n"+ "What is the protocol for managing sepsis in a critical care unit?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care setting:\n\n1. Early recognition: Identify sepsis early by recognizing the signs and symptoms, which may include fever or hypothermia, tachycardia or bradycardia, respiratory distress, altered mental status, and lactic acidosis. Use validated scoring systems like Sequential Organ Failure Assessment (SOFA) score or Quick'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [14]:
user_input = system_prompt+"\n"+ "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the large intestine. The most common symptoms include:\n\n1. Sudden pain in the lower right abdomen, which may start as mild and then gradually worsens over hours.\n2. Loss of appetite and feeling sick to your stomach (nausea).\n3. Fever, often above 101 degrees Fahrenheit (38.3 degrees Celsius).\n4. Vomiting.\n5. Constipation or diarrhea'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [15]:
user_input = system_prompt+"\n"+ "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that results in the loss of hair from specific areas of the scalp or other parts of the body. The exact cause of this condition is not fully understood, but it's believed to be related to a problem with the immune system.\n\nThere are several treatments for addressing sudden patchy hair loss:\n\n1. Corticosteroids: These medications can help reduce inflammation and suppress the immune response that causes hair loss. They can be applied topically or taken orally.\n2"

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [16]:
user_input = system_prompt+"\n"+ "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nThe treatment for a brain injury can depend on the severity and location of the injury. Here are some common treatments:\n\n1. Emergency care: For severe brain injuries, emergency care is crucial. This may include surgery to remove hematomas (clots) or repair skull fractures, as well as measures to maintain adequate blood pressure and oxygen levels.\n\n2. Medications: Depending on the symptoms, various medications may be prescribed. For example, diuretics can help reduce swelling in the brain, while anticonvulsants can prevent seizures.\n\n3. Re'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [17]:
user_input = system_prompt+"\n"+ "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nPrecautions for a person with a fractured leg after a hiking accident include:\n1. Immobilization: The leg should be immobilized using a splint or cast to prevent any further damage or movement that could worsen the injury.\n2. Elevation: Keeping the injured leg elevated above heart level can help reduce swelling and pain.\n3. Rest: Avoid putting weight on the injured leg and try to limit movement as much as possible to allow for proper healing.\n4. Proper nutrition: Adequate intake of nutrients is essential for bone'

In [18]:
user_input = "Does iPhone usage cause sepsis?"
response(user_input)

Llama.generate: prefix-match hit


" No, there is no scientific evidence to support the claim that using an iPhone or any other smartphone can cause sepsis. Sepsis is a serious condition caused by the body's response to infection, and it requires medical attention if suspected. Using a smartphone does not increase the risk of developing sepsis."

Inference :
1. The Persona for the AI is set in the system_prompt although there is no content provided by the program to reference. it would still look at the outside resources to find answer but this time would be looking out, wearing hat of highly knowledgeable and helpful AI medical assistant.
Cons:
* Still there is no way to confirm the percentage of truthfulness from the response of the Model.

## Data Preparation for RAG

In [19]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

### Loading the Data

To DO:
1. mount the drive
2. Load the reference merk manual provided for the problem statement.

*   utilize the provide pdf , use PyMuPDFLOader utility to load and parse PDF documents.




In [21]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
reference_manual_path = "/content/drive/MyDrive/Colab Notebooks/RAG_Module/medical_diagnosis_manual.pdf"

In [23]:
pdf_loader = PyMuPDFLoader(reference_manual_path)

In [24]:
manual = pdf_loader.load()

# Inference:
* loading the provided merck module and utilzing the pdf loader.


### Data Overview

#### Checking the first 5 pages

In [25]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

Page Number : 1
ramya.nandhan@gmail.com
T6VNSBIMPO
This file is meant for personal use by ramya.nandhan@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
ramya.nandhan@gmail.com
T6VNSBIMPO
This file is meant for personal use by ramya.nandhan@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .......................................................................................................................

#### Checking the number of pages

In [26]:
len(manual)

4114

### Data Chunking

In [27]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512, # using size 512
    chunk_overlap= 20 # using 20 to overlap between the chunks
)

In [28]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [29]:
len(document_chunks)

8469

In [30]:
document_chunks[0].page_content

'ramya.nandhan@gmail.com\nT6VNSBIMPO\nThis file is meant for personal use by ramya.nandhan@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [31]:
document_chunks[2].page_content

'Table of Contents\n1\nFront    ................................................................................................................................................................................................................\n1\nCover    .......................................................................................................................................................................................................\n2\nFront Matter    ...........................................................................................................................................................................................\n53\n1 - Nutritional Disorders    ...............................................................................................................................................................\n53\nChapter 1. Nutrition: General Considerations    ...........................................................................................

In [32]:
document_chunks[3].page_content

"Chapter 24. Testing for Hepatic & Biliary Disorders    ......................................................................................................\n305\nChapter 25. Drugs & the Liver    ................................................................................................................................................\n308\nChapter 26. Alcoholic Liver Disease    ....................................................................................................................................\n314\nChapter 27. Fibrosis & Cirrhosis    ............................................................................................................................................\n322\nChapter 28. Hepatitis    ..................................................................................................................................................................\n333\nChapter 29. Vascular Disorders of the Liver    .................................................

### Embedding

In [33]:
embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")

/tmp/ipykernel_6837/3102610685.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [34]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [35]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

In [36]:
embedding_1,embedding_2

([-0.030705919489264488,
  -0.014358802698552608,
  -0.007253531366586685,
  -0.002831835765391588,
  -0.007390058599412441,
  -0.022746413946151733,
  -0.004923881031572819,
  0.03994280472397804,
  0.002664069877937436,
  0.016284577548503876,
  0.016990505158901215,
  0.006889662239700556,
  0.011559619568288326,
  -0.042358215898275375,
  -0.014005579054355621,
  -0.012616854161024094,
  0.002980356104671955,
  -0.03503033146262169,
  -0.00703765545040369,
  -0.027174144983291626,
  -0.0013722102157771587,
  0.016828658059239388,
  -0.09160126000642776,
  -0.02706059068441391,
  -0.002556566847488284,
  0.03095180355012417,
  0.027746744453907013,
  0.006007240153849125,
  0.06740087270736694,
  0.03471415117383003,
  -0.002752357628196478,
  0.024666568264365196,
  0.04707905650138855,
  -0.03178210183978081,
  -0.020140964537858963,
  -0.013932846486568451,
  0.03596018627285957,
  -0.025313926860690117,
  -0.022692756727337837,
  -0.04702921584248543,
  -0.0011480534449219704,
 

** The embedding model provides a fixed-length vector for any number of chunks.
** This is necessary because we want to compare them for similarity.

# Observation :

* The RecursiveCharacterTextSplitter.from_tiktoken_encoder is used for chunking.
* The configuration includes:
encoding_name='cl100k_base'
chunk_size=512
chunk_overlap=20
After splitting the document, 8469 document chunks were created (len(document_chunks)). The initial pages containing user/copyright information are also included in the chunks.

*  The embedding model is initialized using SentenceTransformerEmbeddings.
*  The model_name specified is "thenlper/gte-large".
*  This model produces embedding vectors of length 1024, as confirmed by len(embedding_1) and len(embedding_2).

### Vector Database

In [37]:
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [38]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [39]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

/tmp/ipykernel_6837/2756559696.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


In [40]:
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [41]:
vectorstore.similarity_search("Merck details sepsis",k=3)

[Document(metadata={'creator': 'Atop CHM to PDF Converter', 'file_path': '/content/drive/MyDrive/Colab Notebooks/RAG_Module/medical_diagnosis_manual.pdf', 'modDate': 'D:20260306033201Z', 'author': '', 'creationDate': 'D:20120615054440Z', 'creationdate': '2012-06-15T05:44:40+00:00', 'format': 'PDF 1.7', 'page': 2456, 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'subject': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'moddate': '2026-03-06T03:32:01+00:00', 'source': '/content/drive/MyDrive/Colab Notebooks/RAG_Module/medical_diagnosis_manual.pdf', 'keywords': '', 'trapped': '', 'total_pages': 4114}, page_content='glucose measurement.\nCorticosteroid therapy seems beneficial. Treatment is with replacement doses rather than pharmacologic\ndoses. One regimen consists of hydrocortisone 50 mg IV q 6 h (or 100 mg q 8 h) plus fludrocortisone 50\nμg po once/day during hemodynamic instability and for 3 days thereafter.\nActivated protein C (drotrecogin alfa)

**Conclusion:**
From the retrieved chunks, we observe that all the chunks are related to the key terms [ 'Merck', 'details', 'sepsis' ].

In [42]:
vectorstore.similarity_search("iPhone joke",k=2) # would like to validate if the search picks this similarity search vector from the merck manual.

[Document(metadata={'trapped': '', 'creator': 'Atop CHM to PDF Converter', 'keywords': '', 'modDate': 'D:20260306033201Z', 'total_pages': 4114, 'format': 'PDF 1.7', 'creationdate': '2012-06-15T05:44:40+00:00', 'file_path': '/content/drive/MyDrive/Colab Notebooks/RAG_Module/medical_diagnosis_manual.pdf', 'subject': '', 'creationDate': 'D:20120615054440Z', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'page': 207, 'source': '/content/drive/MyDrive/Colab Notebooks/RAG_Module/medical_diagnosis_manual.pdf', 'author': '', 'moddate': '2026-03-06T03:32:01+00:00'}, page_content='Chapter 14. Bezoars & Foreign Bodies\n198\nramya.nandhan@gmail.com\nT6VNSBIMPO\nThis file is meant for personal use by ramya.nandhan@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'),
 Document(metadata={'subject': '', 'author': '', 'format': 'PDF 1.7', 'keywords': '', 'file_path': '/content/dr

### Retriever

# Inference:
* The retriever is configured using vectorstore.as_retriever.
* The search_type is set to 'similarity', indicating that the retriever will find documents most similar to the query.
* The search_kwargs specify {'k': 2}, meaning the retriever will fetch the top 2 most relevant document chunks based on similarity.

In [43]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2}
)

In [44]:
rel_docs = retriever.get_relevant_documents("What is the protocol for managing sepsis in a critical care unit?")
rel_docs

/tmp/ipykernel_6837/1655728131.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rel_docs = retriever.get_relevant_documents("What is the protocol for managing sepsis in a critical care unit?")


[Document(metadata={'creationDate': 'D:20120615054440Z', 'moddate': '2026-03-06T03:32:01+00:00', 'subject': '', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'keywords': '', 'total_pages': 4114, 'trapped': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-15T05:44:40+00:00', 'modDate': 'D:20260306033201Z', 'author': '', 'page': 2400, 'format': 'PDF 1.7', 'source': '/content/drive/MyDrive/Colab Notebooks/RAG_Module/medical_diagnosis_manual.pdf', 'file_path': '/content/drive/MyDrive/Colab Notebooks/RAG_Module/medical_diagnosis_manual.pdf'}, page_content="16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical,

# **Inference:**
 We can infer that the retriever successfully
identified two relevant document chunks from the medical manual regarding critical care and sepsis management.
  * Both chunks are highly relevant to the query about the protocol for managing sepsis in a critical care unit, indicating that the retrieval mechanism is functioning as expected.

* A directory named medical_db is created using os.makedirs(out_dir) if it does not already exist.
* The Chroma vector store is created from the document_chunks using Chroma.from_documents.
* The embedding_model (thenlper/gte-large) is passed to Chroma for embedding the chunks.
* The persist_directory=out_dir argument ensures that the vector database is saved to the medical_db directory for later use.

In [45]:
llm = Llama(
    model_path=model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


In [46]:
llm("How does we know if a person has sepsis?")['choices'][0]['text']

'\n\nSepsis is a potentially life-threatening condition that can'

Inference:

The response seems generic and appears to be derived from another article. Let's provide our own context and align the response with our needs.

### System and User Prompt Template

To do:
setting the User Prompt template:

**qna_system_message** is the actual system prompt used in the RAG setup to define the AI's role and constraints.
**qna_user_message_template** is the template for the user's input and retrieved context within the RAG setup.

In [47]:
qna_system_message = """
You are an assistant whose work is to review the report and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.

If the answer is not found in the context, respond "I don't know".
"""

In [48]:
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### Response Function

In [49]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

To DO:

1. since the generate_rag_response method is utilizing the chunked 'relevant_document_chunks' , it indicates the response of the RAG wll be based on the merck manual.
2. Applying the userinput to the RAG for each of the 5 questions and finding out the response.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [50]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
print(generate_rag_response(user_input))

Llama.generate: prefix-match hit


Based on the context, the protocol for managing sepsis in a critical care unit includes:
1. Prompt empiric antibiotic therapy: Start immediately after suspecting sepsis.
2. Antibiotic selection: Based on suspected source, clinical setting, knowledge or suspicion of causative organisms and sensitivity patterns common to that specific inpatient unit, and previous culture results.
3. Regimen for septic shock of unknown cause: Gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [51]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(generate_rag_response(user_input))


Llama.generate: prefix-match hit


Based on the context provided, the common symptoms for appendicitis include abdominal pain, anorexia, and abdominal tenderness. Appendicitis cannot be cured via medicine alone; surgery (surgical removal) is required to treat it.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [52]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(generate_rag_response(user_input))

Llama.generate: prefix-match hit


Based on the context provided, the condition being described is alopecia areata. The treatment options for this condition include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). The cause of alopecia areata is believed to be an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers.


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [53]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(generate_rag_response(user_input))

Llama.generate: prefix-match hit


The context mentions that the initial treatment for traumatic brain injury (TBI) includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed for patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation. There is no specific


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [54]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(generate_rag_response(user_input))

Llama.generate: prefix-match hit


Based on the context provided, here's the answer:

The person with a fractured leg should follow these steps:
1. Elevate the injured limb above heart level for the first 2 days to minimize swelling.
2. Apply warmth (using a heating pad) for 15 to 20 minutes after the first 48 hours to relieve pain and speed healing.
3. Immobilize the joints proximal and distal to the injury using elastic bandages, a cast, or a splint.
   - If a cast is used, keep


In [55]:
user_input = "Does iPhone usage cause sepsis?"
print(generate_rag_response(user_input))

Llama.generate: prefix-match hit


" No, there is no scientific evidence to support the claim that using an iPhone or any other smartphone can cause sepsis. Sepsis is a serious condition caused by the body's response to infection, and it requires medical attention if suspected. Using a smartphone does not increase the risk of developing sepsis."

Observation / Inference:

1. The RAG-enabled LLM delivered significantly more accurate, detailed, and medically specific answers. For instance, in Query 1 (Sepsis), it provided explicit antibiotic therapy regimens and selection criteria, which were absent in the standalone LLM's response.
2. By retrieving relevant document chunks, the RAG system ensured that the answers were comprehensive and directly addressed all aspects of the user's query using the provided medical manual. Query 2 (Appendicitis) clearly stated surgical removal as the treatment, and Query 5 (Leg Fracture) offered precise precautions and treatment steps.
3. The answers were clear, concise, and left little room for misinterpretation, which is crucial in medical information retrieval.

### Fine-tuning

In [56]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?."
generate_rag_response(user_input, max_tokens=30)

Llama.generate: prefix-match hit


"Based on the context provided, here's the answer:\n\nThe person with a fractured leg should follow these steps:\n1."

**Inference :**
* Even if the max_tokens is set to 30, the model still didn't generate that
many, as the query could be answered with a limited number of tokens.

* One of the reasons could be that the temperature is set to 0, making the model more deterministic and less creative.

In [57]:
user_input_2 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?."
generate_rag_response(user_input_2, temperature=0.1, max_tokens=350)

Llama.generate: prefix-match hit


"Based on the context provided, here's the answer:\n\nThe person with a fractured leg should follow these steps:\n1. Elevate the injured limb above heart level for the first 2 days to minimize swelling.\n2. Apply warmth (using a heating pad) for 15 to 20 minutes after the first 48 hours to relieve pain and speed healing.\n3. Immobilize the joints proximal and distal to the injury using elastic bandages, a cast, or a splint.\n   - If a cast is used, keep it dry, never put any object inside it, inspect the edges and skin around it daily, apply lotion to red or sore areas, and pad any rough edges with soft material to prevent discomfort from the cast's edges.\n   - Seek medical care immediately if an odor emanates from within the cast or a fever develops, as these may indicate infection.\n4. If possible, apply ice to the injury for pain relief and to reduce swelling.\n5. For fractures that require prolonged immobilization (more than 3-4 weeks), be aware of potential complications such as 

Observation:
1. now with the increased temperature as 0.1 and max_tokens increased , the model response got creative with response.

In [58]:
user_input = "who is the author of this article?."
generate_rag_response(user_input, max_tokens=30)

Llama.generate: prefix-match hit


"I don't know. The context does not provide information about the author of the article."

In [59]:
user_input = "Does iPhone usage cause sepsis?"
generate_rag_response(user_input, max_tokens=30)

Llama.generate: prefix-match hit


" No, there is no scientific evidence to support the claim that using an iPhone or any other smartphone can cause sepsis. Sepsis is a serious condition caused by the body's response to infection, and it requires medical attention if suspected. Using a smartphone does not increase the risk of developing sepsis."

Observation:

now with the a question thta is not in the manual, the model was truthful and crisp on the reply

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

Inference:
due to increasng the temperature the model is proving its creativity.

In [60]:
groundedness_rater_system_message = """You are an expert evaluator assessing the 'groundedness' of an AI-generated answer. Your task is to determine if all factual claims made in the provided 'Answer' are directly and explicitly supported by the 'Context' provided. Do not use any outside knowledge.

Rate the groundedness of the Answer on a scale of 0 to 1, where:
0: The Answer contains significant information not found in the Context, or directly contradicts the Context.
0.5: The Answer contains some information not found in the Context, or is partially supported.
1: All information in the Answer is directly and explicitly supported by the Context.

Provide only the numerical score."""

In [61]:
relevance_rater_system_message = """You are an expert evaluator assessing the 'relevance' of an AI-generated answer to a given question. Your task is to determine if the 'Answer' directly and comprehensively addresses the 'Question' asked. Focus solely on the relationship between the Question and the Answer.

Rate the relevance of the Answer on a scale of 0 to 1, where:
0: The Answer is completely irrelevant to the Question.
0.5: The Answer is partially relevant or only addresses a part of the Question.
1: The Answer is perfectly relevant and fully addresses the Question.

Provide only the numerical score."""

In [62]:
user_message_template = """Context:
{context}

Question:
{question}

Answer:
{answer}

Based on the provided Context, Question, and Answer, evaluate the Groundedness of the Answer."""

In [63]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [64]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
ground,rel = generate_ground_relevance_response(user_input,max_tokens=350)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 1. The Answer directly and explicitly supports all factual claims made in the Context regarding managing sepsis in a critical care unit, including administering antibiotics, draining abscesses, normalizing blood glucose, using corticosteroids, and considering activated protein C therapy.

 1. The Answer directly addresses the Question by providing a comprehensive list of protocols for managing sepsis in a critical care unit, including administration of antibiotics, drainage of abscesses and excision of necrotic tissues, normalization of blood glucose, corticosteroid therapy, and emerging therapies.


In [65]:
five_questions = [
    "What is the protocol for managing sepsis in a critical care unit?",
    "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
]

for i, question in enumerate(five_questions):
    print(f"\n--- Question {i+1}: {question} ---")
    ground,rel = generate_ground_relevance_response(question,max_tokens=350)
    print(ground,end="\n\n")
print(rel)
    #print(f"Answer: {answer}")
#    print("\n" + "="*80 + "\n")


--- Question 1: What is the protocol for managing sepsis in a critical care unit? ---


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 1. The Answer directly and explicitly supports all factual claims made in the Context regarding managing sepsis in a critical care unit, including administering antibiotics, draining abscesses, normalizing blood glucose, using corticosteroids, and considering activated protein C therapy.


--- Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it? ---


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 1. The symptoms mentioned in the Answer (abdominal pain, anorexia, and abdominal tenderness) are directly supported by the Context as common symptoms of appendicitis.
2. The statement that appendicitis cannot be cured via medicine alone is also directly supported by the Context, which states that antibiotics improve survival but do not cure the condition and that surgery is the standard treatment.

Therefore, the Answer is fully grounded in the Context and receives a score of 1.


--- Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it? ---


...

 1. The Answer directly and explicitly supports the fact that sudden patchy hair loss, or alopecia areata, can be treated with various methods including topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). It also mentions that oral antimalarials may be used for certain conditions.
2. The Answer correctly states that the cause of sudden patchy hair loss is believed to be an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers, which is consistent with the information provided in the Context.

Therefore, based on the given Context and Question, the Answer can be rated as 1 for groundedness.


--- Question 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function? ---


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 1.0. The Answer directly and explicitly repeats information from the Context regarding the recommended treatments for traumatic brain injury (TBI), including supportive care measures such as maintaining adequate ventilation, oxygenation, and blood pressure, potential need for surgery, prevention of complications in the first few days, rehabilitation, and prevention of systemic complications. The Answer also correctly states that there is no specific treatment for TBI but supportive care should be provided, and mentions the possibility of seizures and drug treatment accordingly, all of which are directly supported by the Context.


--- Question 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery? ---


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 1. The Answer directly and explicitly supports all factual claims made in the Context regarding the necessary precautions and treatment steps for a person with a fractured leg, including keeping the area dry, inspecting the cast regularly, applying lotion to any red or sore areas, seeking medical care if necessary, maintaining good hygiene, immobilizing the injured limb using a cast or splint, and avoiding weight bearing on the injured leg. The Answer also correctly states that periodic application of warmth may be helpful for pain relief after 48 hours and that severe swelling may require bivalving the cast. Additionally, the Answer accurately explains the potential complications of prolonged immobilization and the importance of following the recommended care and recovery plan closely.

 1. The Answer directly addresses the necessary precautions and treatment steps for a person with a fractured leg, as well as considerations for their care and recovery, making it perfectly relevant a

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [66]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
ground,rel = generate_ground_relevance_response(user_input_5,max_tokens=500)

print(ground,end="\n\n")
print(rel)


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 1. The Answer directly and explicitly supports all factual claims made in the Context regarding the necessary precautions and treatment steps for a person with a fractured leg, including keeping the area dry, inspecting the cast regularly, applying lotion to any red or sore areas, seeking medical care if necessary, maintaining good hygiene, immobilizing the injured limb using a cast or splint, and avoiding weight bearing on the injured leg. The Answer also correctly states that periodic application of warmth may be helpful for pain relief after 48 hours and that severe swelling may require bivalving the cast. Additionally, the Answer accurately explains the potential complications of prolonged immobilization and the importance of following the recommended care and recovery plan closely.

 1. The Answer directly addresses the necessary precautions and treatment steps for a person with a fractured leg, as well as considerations for their care and recovery, making it perfectly relevant a

answers for all the 5 questions are iterated through the Python code and rendered in one output.

# Observation:
* Groundedness Scores: For all five queries, the groundedness scores consistently received a 1. This indicates that the AI-generated answers were entirely and explicitly supported by the context provided by the retriever. The textual explanations confirm that no factual claims were made without direct support from the retrieved medical manual sections, nor were there any contradictions.

* Relevance Scores: Similarly, the relevance scores for all five queries were rated as 1. This signifies that the answers directly and comprehensively addressed the questions asked. The textual explanations highlight that the responses covered all aspects of the user's inquiry, demonstrating the system's ability to extract and synthesize pertinent information effectively.

## Actionable Insights and Business Recommendations

# Actionable Insights:
**LLM response** revealed below features
* General and Less Specific,
* Potential for Generality/Hallucination,
* Incomplete Information
* Lack of Factual Grounding

**RAG enabled LLM Response**s were proving
* **High Accuracy and Specificity:** The RAG-enabled LLM delivered significantly more accurate, detailed, and medically specific answers. For instance, in Query 1 (Sepsis), it provided explicit antibiotic therapy regimens and selection criteria, which were absent in the standalone LLM's response.

* **Contextual Completeness:** By retrieving relevant document chunks, the RAG system ensured that the answers were comprehensive and directly addressed all aspects of the user's query using the provided medical manual. Query 2 (Appendicitis) clearly stated surgical removal as the treatment, and Query 5 (Leg Fracture) offered precise precautions and treatment steps.

* **Factual Grounding:** Every response generated by the RAG system was explicitly linked to the content of the medical_diagnosis_manual.pdf. The LLM-as-a-judge evaluation (groundedness and relevance scores of 1) further validates that the answers are directly supported by the context.

* Reduced Ambiguity: The answers were clear, concise, and left little room for misinterpretation, which is crucial in medical information retrieval.

**  **Impact of Context retrieval and Prompt Engineering ** **


* **Guided Response Style:** The qna_system_message (that directs the AI to respond only using the context provided and to state 'I don't know' if the answer is not found) and the qna_user_message_template (that structures the context and question for the LLM) were highly effective. They forced the RAG-enabled LLM to adhere strictly to the provided medical manual, preventing it from generating ungrounded or speculative information. This adherence is critical for medical applications where accuracy and reliability are paramount.

# Business Recommendations:

* Although the  medical assistant would lookup on the Merck manual before suggesting to the medical practitioner, it would be more beneficial to pick up real time information rather than referencing from a static manual. real time information provided by the AI assistant will be needed.
* getting the facts of the patient as inputs and basing the questions accordingly will be helpful, so may be having an interactive app that can take in the inputs of the patient / collecting background of the request will be helpful to nail down to exact problem /question that the AI can find out answer to.

<font size=6 color='blue'>Power Ahead</font>
___